# HDBSCAN Finetuning — Original vs Pruned Clusters

Fine-tunes a MiniLM model on hard HDBSCAN cluster assignments using
`hardfinetuning_crossentropy.py` from the RecsysUpgrade repo.

Runs two comparisons back-to-back:
1. **Original HDBSCAN** — all non-noise clusters from `hdbscan_clusters.csv`
2. **Pruned HDBSCAN** — LSH-pruned clusters from `hdbscan_clusters_pruned.csv`

Cluster IDs are remapped to contiguous 0…N-1 integers before training
(noise points, label = −1, are excluded from both runs).

Paths and cluster files are produced by `UMAP_HDBSCAN.ipynb`.


### Step 1: Setup (drive mounting, paths)


In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE          = "/content/drive/MyDrive/S2026/Spotify Playlist Data/Processed Data/calced_embeddings"
PROJECT_BASE        = "/content/drive/MyDrive/S2026/CS 274/Playlist-Recommender"
EMBEDDINGS_PKL      = f"{DRIVE_BASE}/mean_pooled_embeddings.pkl"
CLUSTERS_OUTPUT_DIR = f"{PROJECT_BASE}/umap clusters"

FINETUNED_MODEL_DIR         = f"{PROJECT_BASE}/hdbscan_finetuned_model"
FINETUNED_PRUNED_MODEL_DIR  = f"{PROJECT_BASE}/hdbscan_finetuned_model_pruned"
REPO_DIR                    = "/content/RecsysUpgrade"

os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
os.makedirs(FINETUNED_PRUNED_MODEL_DIR, exist_ok=True)
print("Paths configured.")


Mounted at /content/drive
Paths configured.


### Step 2: Clone repo and install requirements


In [3]:
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/siddmohanty111/RecsysUpgrade.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)


Cloning into '/content/RecsysUpgrade'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 141 (delta 73), reused 80 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 2.47 MiB | 39.45 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Repo ready: /content/RecsysUpgrade


In [4]:
%pip install -r {REPO_DIR}/requirements.txt -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### Step 3: Imports


In [5]:
import importlib.util
import pickle

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


### Step 4: Load finetuning module and playlist titles


In [ ]:
ft_hard_path = os.path.join(
    REPO_DIR, "src", "finetuning", "hardfinetuning_crossentropy.py"
)
spec    = importlib.util.spec_from_file_location("hardfinetuning_crossentropy", ft_hard_path)
hard_ft = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hard_ft)
print("hardfinetuning_crossentropy loaded.")

# Load playlist titles from the embeddings pkl (same source as UMAP_HDBSCAN.ipynb)
with open(EMBEDDINGS_PKL, "rb") as f:
    raw_data = pickle.load(f)

playlist_titles = (
    raw_data.get("playlist_titles", {})
    if isinstance(raw_data, dict)
    else {}
)
print(f"Playlist titles available: {len(playlist_titles)}")


hardfinetuning_crossentropy loaded.
Playlist titles available: 1000001


---
## Part 1: Original HDBSCAN Clusters

Fine-tune on the full HDBSCAN cluster assignments (`hdbscan_clusters.csv`).
Noise points (cluster = −1) are removed. Cluster IDs are remapped to 0…N-1.

After some testing, I identified that the finetuned models began to overfit


#### Load and prepare original HDBSCAN cluster dataset


In [7]:
clusters_df = pd.read_csv(
    os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_clusters_initial.csv"),
    dtype={"pid": str},
)
print(f"Loaded {len(clusters_df)} rows. Unique clusters (incl. noise): {clusters_df['cluster'].nunique()}")

# Remove noise points (HDBSCAN labels noise as -1)
clusters_df = clusters_df[clusters_df["cluster"] != -1].reset_index(drop=True)
print(f"After removing noise: {len(clusters_df)} playlists, {clusters_df['cluster'].nunique()} clusters")

# Remap cluster IDs to contiguous 0…N-1
unique_ids  = sorted(clusters_df["cluster"].unique())
id_map      = {old: new for new, old in enumerate(unique_ids)}
clusters_df["cluster"] = clusters_df["cluster"].map(id_map)
num_clusters_orig = clusters_df["cluster"].nunique()
print(f"Cluster IDs remapped to 0…{num_clusters_orig - 1}")

# Build dataset
data_df = pd.DataFrame({
    "Playlist Title": [playlist_titles.get(pid, "") for pid in clusters_df["pid"]],
    "Cluster Labels": clusters_df["cluster"],
})
data_df = data_df[data_df["Playlist Title"].str.strip() != ""].reset_index(drop=True)
print(f"Dataset after dropping untitled playlists: {len(data_df)}")

# 90 / 10 train–val split
train_df, val_df = train_test_split(data_df, test_size=0.1, random_state=42)

TRAIN_CSV = os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_train.csv")
VAL_CSV   = os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_val.csv")

train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV,   index=False)
print(f"Train: {len(train_df)}  Val: {len(val_df)}")
display(data_df.head())


Loaded 1000001 rows. Unique clusters (incl. noise): 134
After removing noise: 339800 playlists, 133 clusters
Cluster IDs remapped to 0…132
Dataset after dropping untitled playlists: 339800
Train: 305820  Val: 33980


,Playlist Title,Cluster Labels
0,name,96
1,Throwbacks,109
2,korean,17
3,BOP,104
4,old country,36


In [8]:
hard_ft.run(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    output_dir=FINETUNED_MODEL_DIR,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=128,
    epochs=20,
    learning_rate=3e-4,
    warmup_steps=200,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to /content/drive/MyDrive/S2026/CS 274/Playlist-Recommender/hdbscan_finetuned_model


---
## Part 2: Pruned HDBSCAN Clusters

Fine-tune on the LSH-pruned cluster assignments (`hdbscan_clusters_pruned.csv`).
Overly broad clusters have already been removed by `lsh_cluster_picking.prune_clusters`.
Cluster IDs are remapped to 0…N-1 again since pruning makes them non-contiguous.


#### Load and prepare pruned HDBSCAN cluster dataset


In [9]:
clusters_pruned_df = pd.read_csv(
    os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_clusters_pruned_initial.csv"),
    dtype={"pid": str},
)
print(f"Loaded {len(clusters_pruned_df)} rows, {clusters_pruned_df['cluster'].nunique()} clusters (post-pruning)")

# Remap cluster IDs to contiguous 0…N-1
unique_ids_pruned  = sorted(clusters_pruned_df["cluster"].unique())
id_map_pruned      = {old: new for new, old in enumerate(unique_ids_pruned)}
clusters_pruned_df["cluster"] = clusters_pruned_df["cluster"].map(id_map_pruned)
num_clusters_pruned = clusters_pruned_df["cluster"].nunique()
print(f"Cluster IDs remapped to 0…{num_clusters_pruned - 1}")

# Build dataset
data_pruned_df = pd.DataFrame({
    "Playlist Title": [playlist_titles.get(pid, "") for pid in clusters_pruned_df["pid"]],
    "Cluster Labels": clusters_pruned_df["cluster"],
})
data_pruned_df = data_pruned_df[data_pruned_df["Playlist Title"].str.strip() != ""].reset_index(drop=True)
print(f"Dataset after dropping untitled playlists: {len(data_pruned_df)}")

# 90 / 10 train–val split
train_pruned_df, val_pruned_df = train_test_split(data_pruned_df, test_size=0.1, random_state=42)

TRAIN_PRUNED_CSV = os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_train_pruned.csv")
VAL_PRUNED_CSV   = os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_val_pruned.csv")

train_pruned_df.to_csv(TRAIN_PRUNED_CSV, index=False)
val_pruned_df.to_csv(VAL_PRUNED_CSV,   index=False)
print(f"Train: {len(train_pruned_df)}  Val: {len(val_pruned_df)}")
display(data_pruned_df.head())


Loaded 338246 rows, 130 clusters (post-pruning)
Cluster IDs remapped to 0…129
Dataset after dropping untitled playlists: 338246
Train: 304421  Val: 33825


,Playlist Title,Cluster Labels
0,name,93
1,Throwbacks,106
2,korean,17
3,BOP,101
4,old country,36


In [10]:
hard_ft.run(
    train_csv=TRAIN_PRUNED_CSV,
    val_csv=VAL_PRUNED_CSV,
    output_dir=FINETUNED_PRUNED_MODEL_DIR,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=128,
    epochs=20,
    learning_rate=3e-4,
    warmup_steps=200,
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/304421 [00:00<?, ? examples/s]

Map:   0%|          | 0/33825 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,2.530431,2.314075,0.587317
2,2.277381,2.270408,0.594087
3,2.232451,2.249701,0.599172
4,2.205051,2.243198,0.598492
5,2.183403,2.236693,0.600976
6,2.165470,2.242097,0.601153
7,2.148890,2.239102,0.599734
8,2.133005,2.251275,0.598788
9,2.118213,2.243088,0.601567
10,2.104566,2.247701,0.600650


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to /content/drive/MyDrive/S2026/CS 274/Playlist-Recommender/hdbscan_finetuned_model_pruned
